# Leaderboard — compare quality and operations

Build a reproducible leaderboard from evaluator aggregates while keeping Stirrup tools-only and code tracks separate.

**Tutorial contract:** run cells from top to bottom. Every external dependency is checked before use,
outputs go under `artifacts/kdd_tutorial/`, and no credential value is printed.


In [ ]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()
if repo is None:
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")
ENV_FILE = repo / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
print("environment source:", ENV_FILE)

In [ ]:
import pandas as pd
REPORT_ROOT = ARTIFACTS / "reports"
aggregate_files = sorted(REPORT_ROOT.glob("**/_aggregate.json"))
print("aggregates:", len(aggregate_files))


## 1. Load evaluator outputs

Folder convention: name report folders like `stirrup_tools`, `stirrup_code`, `openai_tools`, etc.


In [ ]:
rows = []
for path in aggregate_files:
    report = json.loads(path.read_text())
    folder = path.parent.name
    inferred_track = "code" if "code" in folder.lower() else "tools-only"
    totals, ops = report.get("totals", {}), report.get("ops", {})
    for runner in report.get("runners", ["unknown"]):
        for model in report.get("models", ["unknown"]):
            rows.append({"track": inferred_track, "runner": runner, "model": model, "scenarios": totals.get("scenarios", 0), "pass_rate": totals.get("pass_rate", 0.0), "tool_calls": ops.get("tool_calls_total", 0), "tokens": ops.get("tokens_in_total", 0) + ops.get("tokens_out_total", 0), "latency_p50_ms": ops.get("duration_ms_p50"), "est_cost_usd": ops.get("est_cost_usd_total", 0.0), "source": str(path.relative_to(REPO))})
board = pd.DataFrame(rows)
board


## 2. Rank within track

Pass rate is primary. Cost and median latency break ties; never rank code-enabled Stirrup against tools-only agents as if they had the same capabilities.


In [ ]:
if not board.empty:
    leaderboard = board.sort_values(["track", "pass_rate", "est_cost_usd", "latency_p50_ms"], ascending=[True, False, True, True], na_position="last").reset_index(drop=True)
    leaderboard["rank_in_track"] = leaderboard.groupby("track").cumcount() + 1
    leaderboard = leaderboard[["track", "rank_in_track", "runner", "model", "scenarios", "pass_rate", "est_cost_usd", "latency_p50_ms", "tool_calls", "tokens", "source"]]
    display(leaderboard)
else:
    leaderboard = board
    print("No aggregates yet. Run notebook 08 for each runner/model configuration.")


## 3. Export the auditable table


In [ ]:
LEADERBOARD_CSV = ARTIFACTS / "kdd_leaderboard.csv"
leaderboard.to_csv(LEADERBOARD_CSV, index=False)
LEADERBOARD_CSV


## Reporting checklist

Publish the scenario-set version, execution model, judge model, agent runner, track, sample count, pass rate, latency, tokens, estimated cost, and the aggregate file used for every row.
